# Análisis de ventas — limpieza y preparación para Power BI

**Empresa:** Timaran  
**Consultor:** Faro Consultores y Asesores S.A.S  
**Fecha:** 06 de junio de 2026

Este notebook prepara el archivo de ventas para construir informes en Power BI.  
El proceso conserva las validaciones principales para revisar estructura, nulos, vacíos, productos sin código, columnas no numéricas, registros no clasificados y calidad final de la tabla.

El flujo general es:

1. Cargar el reporte de ventas y la tabla maestra de productos.
2. Normalizar encabezados y códigos de producto.
3. Limpiar la estructura original del reporte.
4. Validar nulos, vacíos y tipos de datos.
5. Convertir la tabla de ventas de formato ancho a formato largo.
6. Eliminar ventas nulas, negativas o en cero.
7. Cruzar las ventas con la tabla maestra.
8. Separar productos no clasificados para revisión.
9. Generar el archivo final para Power BI.


## 1. Importar librerías y definir rutas

En esta sección se importan las librerías necesarias y se definen las rutas de entrada y salida.

Archivos usados:

- `Tabla Analisis de ventas.xlsx`: reporte original de ventas.
- `Tabla_Maestra_Productos.xlsx`: tabla con códigos, nombres y clasificaciones de productos.
- `Ventas_Limpias_PowerBI.xlsx`: archivo final limpio para cargar en Power BI.


In [312]:
import pandas as pd
from pathlib import Path
import re
import unicodedata

# Rutas principales del proyecto
DATA_DIR = Path("../Data")
RESULTADOS_DIR = Path("../Resultado")
CARPETA_SALIDA_MENSUAL = Path("../Informes Limpios")
INFORME_A_LIMPIAR = Path("PRODUCTOS 2026.xlsx")

ruta_reporte_ventas = DATA_DIR / INFORME_A_LIMPIAR
ruta_tabla_maestra = DATA_DIR / "Tabla_Maestra_Productos.xlsx"
ruta_salida_powerbi = RESULTADOS_DIR / "Ventas_Limpias_PowerBI.xlsx"
ruta_productos_no_clasificados = DATA_DIR / "Tabla_Productos_No_Clasificados.xlsx"

# Crear carpetas de salida si todavía no existen
RESULTADOS_DIR.mkdir(parents=True, exist_ok=True)
CARPETA_SALIDA_MENSUAL.mkdir(parents=True, exist_ok=True)

# Columna que identifica el periodo mensual de facturación
COLUMNA_PERIODO = "PERIODO"

# Seguridad:
# False = si ya existe un Excel del mismo periodo, el notebook se detiene.
# True  = permite reemplazar el Excel mensual existente.
PERMITIR_SOBREESCRITURA = True

## 2. Cargar y normalizar la tabla maestra

La tabla maestra es la base para clasificar los productos vendidos.  
Aquí se normalizan los encabezados y el campo `CODIGO_PRODUCTO` para evitar errores en el cruce con las ventas.


In [313]:
df_maestro = pd.read_excel(
    ruta_tabla_maestra,
    sheet_name="Productos"
)

# Normalizar encabezados: quitar espacios, pasar a mayúsculas y corregir espacios dobles
df_maestro.columns = (
    df_maestro.columns
    .str.strip()
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
)

# Validar que exista la columna clave para el cruce
if "CODIGO_PRODUCTO" not in df_maestro.columns:
    raise ValueError("La tabla maestra debe tener una columna llamada CODIGO_PRODUCTO.")

# Asegurar que el código quede como texto limpio y sin espacios internos
df_maestro["CODIGO_PRODUCTO"] = (
    df_maestro["CODIGO_PRODUCTO"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", "", regex=True)
)

display(df_maestro.head())


,CODIGO_PRODUCTO,NOMBRE_PRODUCTO,CLASIFICACION I,CLASIFICACION II,CLASIFICACION III,CLASIFICACION IV
0,2542,ENVASE SADMAN TRANSP 105 ML PLATA CROMADO X 54...,ENVASE,SADMAN,105 ML,54
1,2610,ENVASE SALVAJE 100 ML FUCSIA X 72 UND,ENVASE,SALVAJE,100 ML,72
2,2608,ENVASE SALVAJE 100 ML NEGRO X 72 UND,ENVASE,SALVAJE,100 ML,72
3,2229,180 FUNDAS SURTIDAS X 6 COLORES,FUNDA,PAQUETE,NO APLICA,NO APLICA
4,2278,A MILKSHAKE PLEASE DM,ESENCIA,ESENCIA NICHO,ARMAF,DM


### Validación de estructura de la tabla maestra

Antes de continuar, se revisa que las columnas necesarias para el informe existan en la tabla maestra.  
Esto ayuda a detectar cambios en el archivo de origen antes de ejecutar todo el proceso.


In [314]:
columnas_maestro_esperadas = [
    "CODIGO_PRODUCTO",
    "NOMBRE_PRODUCTO",
    "CLASIFICACION I",
    "CLASIFICACION II",
    "CLASIFICACION III",
    "CLASIFICACION IV"
]

columnas_faltantes_maestro = [
    col for col in columnas_maestro_esperadas
    if col not in df_maestro.columns
]

if columnas_faltantes_maestro:
    raise ValueError(f"Faltan columnas en la tabla maestra: {columnas_faltantes_maestro}")

print("Validación correcta: la tabla maestra contiene las columnas esperadas.")
print("Filas y columnas tabla maestra:", df_maestro.shape)


Validación correcta: la tabla maestra contiene las columnas esperadas.
Filas y columnas tabla maestra: (2502, 6)


## 3. Cargar el reporte original de ventas

El archivo de ventas viene con filas de encabezado adicionales y una columna final de total.  
En este paso se carga la hoja `Análisis de ventas`, se eliminan las filas que no pertenecen al detalle de productos y se quita la última columna porque corresponde al total general.


In [315]:
df_ventas_original = pd.read_excel(
    ruta_reporte_ventas,
    sheet_name="Análisis de ventas",
    header=2,
    index_col=0
)

# Eliminar filas internas del encabezado que no hacen parte de los productos
# Se conservan los mismos pasos de limpieza usados en el notebook original.
df_ventas_original = df_ventas_original.drop(df_ventas_original.index[[0, 1]])

# Identificar y eliminar la última columna, normalmente correspondiente al total general
ultima_columna = df_ventas_original.columns[-1]
df_ventas_original = df_ventas_original.drop(columns=[ultima_columna])

display(df_ventas_original.head(10))


,enero 2026,febrero 2026,marzo 2026,abril 2026,mayo 2026,junio 2026
[2542] ENVASE SADMAN TRANSP 105 ML PLATA CROMADO X 54 UND,NaN,46,NaN,NaN,0,NaN
[2229] 180 FUNDAS SURTIDAS X 6 COLORES,144,133,171,126,163,131
[2278] A MILKSHAKE PLEASE DM,246670,170809,314880,309464,276370,188870
[2834] ACEITE DE CASTOR P40 - FINDET ARH/52,NaN,NaN,NaN,NaN,2000,3110
[0252] ACQUA FRESCA HM,87120,66980,127531,93700,88215,66410
[509] ADDICTIVE VIBE HM,162420,133250,210040,229821,136380,150080
[0723] ADELINE DM,39370,39720,32920,37650,30060,24670
[400] ADIVINA DM,25410,14150,39940,20440,16930,11170
[372] ADREA HM,11060,5449,46330,7210,4360,5700
[291] ADRIAN HM,120440,112535,151520,162040,118608,119860


## 4. Preparar la columna de producto

El nombre del producto viene inicialmente como índice del DataFrame.  
Por eso se reinicia el índice y se renombra la columna como `PRODUCTO`, que será usada para extraer el código del producto.


In [316]:
df_ventas_original = df_ventas_original.reset_index()

df_ventas_original = df_ventas_original.rename(
    columns={"index": "PRODUCTO"}
)

display(df_ventas_original.head())


,PRODUCTO,enero 2026,febrero 2026,marzo 2026,abril 2026,mayo 2026,junio 2026
0,[2542] ENVASE SADMAN TRANSP 105 ML PLATA ...,NaN,46,NaN,NaN,0,NaN
1,[2229] 180 FUNDAS SURTIDAS X 6 COLORES,144,133,171,126,163,131
2,[2278] A MILKSHAKE PLEASE DM,246670,170809,314880,309464,276370,188870
3,[2834] ACEITE DE CASTOR P40 - FINDET ARH/52,NaN,NaN,NaN,NaN,2000,3110
4,[0252] ACQUA FRESCA HM,87120,66980,127531,93700,88215,66410


## 5. Normalizar encabezados del reporte de ventas

Se convierten los nombres de las columnas a mayúsculas para trabajar con una estructura uniforme.  
Esto evita errores por diferencias de escritura entre archivos.


In [317]:
df_ventas_original.columns = df_ventas_original.columns.str.upper()

display(df_ventas_original.head())


,PRODUCTO,ENERO 2026,FEBRERO 2026,MARZO 2026,ABRIL 2026,MAYO 2026,JUNIO 2026
0,[2542] ENVASE SADMAN TRANSP 105 ML PLATA ...,NaN,46,NaN,NaN,0,NaN
1,[2229] 180 FUNDAS SURTIDAS X 6 COLORES,144,133,171,126,163,131
2,[2278] A MILKSHAKE PLEASE DM,246670,170809,314880,309464,276370,188870
3,[2834] ACEITE DE CASTOR P40 - FINDET ARH/52,NaN,NaN,NaN,NaN,2000,3110
4,[0252] ACQUA FRESCA HM,87120,66980,127531,93700,88215,66410


### Validación de tamaño y columnas del reporte

Esta validación permite revisar cuántas filas y columnas tiene el reporte después de la primera limpieza, además de confirmar los nombres de las columnas disponibles.


In [318]:
print("Filas y columnas:", df_ventas_original.shape)

print("Columnas:")
print(df_ventas_original.columns)


Filas y columnas: (2374, 7)
Columnas:
Index(['PRODUCTO', 'ENERO 2026', 'FEBRERO 2026', 'MARZO 2026', 'ABRIL 2026',
       'MAYO 2026', 'JUNIO 2026'],
      dtype='str')


### Validación de valores nulos y vacíos

Se revisa cada columna para identificar:

- Valores nulos (`NaN`).
- Celdas vacías o con solo espacios.
- Total combinado de nulos y vacíos.

Esta revisión es importante antes de transformar los datos, porque permite detectar problemas de calidad en el archivo original.


In [319]:
df_revision_nulls = pd.DataFrame({
    "COLUMNA": df_ventas_original.columns,
    "CANTIDAD_NULL": df_ventas_original.isnull().sum().values,
    "CANTIDAD_VACIOS": [
        df_ventas_original[col]
        .astype("string")
        .str.strip()
        .eq("")
        .sum()
        for col in df_ventas_original.columns
    ]
})

df_revision_nulls["TOTAL_NULL_Y_VACIOS"] = (
    df_revision_nulls["CANTIDAD_NULL"] + df_revision_nulls["CANTIDAD_VACIOS"]
)

display(df_revision_nulls)


,COLUMNA,CANTIDAD_NULL,CANTIDAD_VACIOS,TOTAL_NULL_Y_VACIOS
0,PRODUCTO,0,0,0
1,ENERO 2026,468,0,468
2,FEBRERO 2026,499,0,499
3,MARZO 2026,475,0,475
4,ABRIL 2026,471,0,471
5,MAYO 2026,509,0,509
6,JUNIO 2026,574,0,574


## 6. Limpiar texto de productos

Se limpian los espacios del campo `PRODUCTO`.  
Esto deja los nombres más consistentes y facilita la extracción del código que viene entre corchetes.


In [320]:
if "PRODUCTO" not in df_ventas_original.columns:
    raise ValueError("No se encontró la columna PRODUCTO en el reporte de ventas.")

df_ventas_original["PRODUCTO"] = (
    df_ventas_original["PRODUCTO"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

display(df_ventas_original[["PRODUCTO"]].head(10))


,PRODUCTO
0,[2542] ENVASE SADMAN TRANSP 105 ML PLATA CROMA...
1,[2229] 180 FUNDAS SURTIDAS X 6 COLORES
2,[2278] A MILKSHAKE PLEASE DM
3,[2834] ACEITE DE CASTOR P40 - FINDET ARH/52
4,[0252] ACQUA FRESCA HM
5,[509] ADDICTIVE VIBE HM
6,[0723] ADELINE DM
7,[400] ADIVINA DM
8,[372] ADREA HM
9,[291] ADRIAN HM


## 7. Extraer el código del producto

El código se extrae desde el texto de `PRODUCTO`, tomando el valor que aparece entre corchetes `[]`.  
Este código será la llave para cruzar las ventas con la tabla maestra.


In [321]:
df_ventas_original["CODIGO_PRODUCTO"] = (
    df_ventas_original["PRODUCTO"]
    .str.extract(r"\[(.*?)\]", expand=False)
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", "", regex=True)
)

display(df_ventas_original[["PRODUCTO", "CODIGO_PRODUCTO"]].head(10))


,PRODUCTO,CODIGO_PRODUCTO
0,[2542] ENVASE SADMAN TRANSP 105 ML PLATA CROMA...,2542
1,[2229] 180 FUNDAS SURTIDAS X 6 COLORES,2229
2,[2278] A MILKSHAKE PLEASE DM,2278
3,[2834] ACEITE DE CASTOR P40 - FINDET ARH/52,2834
4,[0252] ACQUA FRESCA HM,0252
5,[509] ADDICTIVE VIBE HM,509
6,[0723] ADELINE DM,0723
7,[400] ADIVINA DM,400
8,[372] ADREA HM,372
9,[291] ADRIAN HM,291


### Validación de productos sin código

Aquí se listan los productos a los que no se les pudo extraer código.  
Estos registros no se pueden cruzar con la tabla maestra, por eso se revisan antes de eliminarlos.


In [322]:
productos_sin_codigo = df_ventas_original[
    df_ventas_original["CODIGO_PRODUCTO"].isna()
]

print("Cantidad de productos sin código:", productos_sin_codigo.shape[0])
display(productos_sin_codigo[["PRODUCTO"]])


Cantidad de productos sin código: 2


,PRODUCTO
799,FLETE INTERNACIONAL
2259,Transporte


### Eliminación de productos sin código

Se eliminan los productos que no tienen `CODIGO_PRODUCTO`, porque no pueden clasificarse correctamente ni cruzarse con la tabla maestra.


In [323]:
df_ventas_original = df_ventas_original[
    df_ventas_original["CODIGO_PRODUCTO"].notna()
].copy()

# Reiniciar índice después de eliminar registros
df_ventas_original = df_ventas_original.reset_index(drop=True)


### Validación posterior a la eliminación de productos sin código

Se vuelve a revisar que no queden productos sin código antes de continuar con la transformación de las ventas.


In [324]:
productos_sin_codigo = df_ventas_original[
    df_ventas_original["CODIGO_PRODUCTO"].isna()
]

print("Cantidad final de productos sin código:", productos_sin_codigo.shape[0])
display(productos_sin_codigo[["PRODUCTO"]])


Cantidad final de productos sin código: 0


,PRODUCTO


## 8. Identificar columnas de meses

Después de preparar `PRODUCTO` y `CODIGO_PRODUCTO`, todas las demás columnas corresponden a periodos o meses de ventas.  
Esta lista se usa para validar y convertir las cantidades facturadas.


In [325]:
columnas_meses = [
    col for col in df_ventas_original.columns
    if col not in ["PRODUCTO", "CODIGO_PRODUCTO"]
]

print("Columnas de meses encontradas:")
print(columnas_meses)

print("Cantidad de columnas de meses:", len(columnas_meses))


Columnas de meses encontradas:
['ENERO 2026', 'FEBRERO 2026', 'MARZO 2026', 'ABRIL 2026', 'MAYO 2026', 'JUNIO 2026']
Cantidad de columnas de meses: 6


### Validación de valores no numéricos en columnas de meses

Antes de convertir los meses a número, se revisa si existen valores que no puedan convertirse correctamente.  
Si aparecen columnas con problemas, el notebook muestra algunos ejemplos para facilitar la corrección del archivo original.


In [326]:
columnas_con_problemas = []

for col in columnas_meses:
    datos_convertidos = pd.to_numeric(df_ventas_original[col], errors="coerce")

    valores_problematicos = df_ventas_original[
        df_ventas_original[col].notna() & datos_convertidos.isna()
    ]

    if len(valores_problematicos) > 0:
        columnas_con_problemas.append(col)
        print(f"Problemas en la columna: {col}")
        display(valores_problematicos[["PRODUCTO", col]].head(10))

print("Columnas con problemas:")
print(columnas_con_problemas)


Columnas con problemas:
[]


### Conversión de columnas de meses a formato numérico

Las columnas de meses se convierten a tipo numérico para que Power BI pueda sumar, filtrar y analizar correctamente las cantidades facturadas.


In [327]:
df_ventas_original[columnas_meses] = df_ventas_original[columnas_meses].apply(
    pd.to_numeric,
    errors="coerce"
)


### Validación de tipos de datos

Se revisa que las columnas de meses hayan quedado como datos numéricos.


In [328]:
display(df_ventas_original[columnas_meses].dtypes)


ENERO 2026      float64
FEBRERO 2026    float64
MARZO 2026      float64
ABRIL 2026      float64
MAYO 2026       float64
JUNIO 2026      float64
dtype: object

### Validación de nulos después de convertir a numérico

Cuando se convierten valores a número, cualquier dato inválido puede convertirse en `NaN`.  
Por eso se revisa nuevamente la cantidad de nulos por cada periodo.


In [329]:
df_nulls_meses = (
    df_ventas_original[columnas_meses]
    .isnull()
    .sum()
    .reset_index()
)

df_nulls_meses.columns = ["MES", "CANTIDAD_NULL"]

display(df_nulls_meses)


,MES,CANTIDAD_NULL
0,ENERO 2026,466
1,FEBRERO 2026,497
2,MARZO 2026,474
3,ABRIL 2026,469
4,MAYO 2026,507
5,JUNIO 2026,572


## 9. Convertir ventas de formato ancho a formato largo

El reporte original tiene una columna por cada periodo.  
Para Power BI es más conveniente tener una tabla en formato largo, donde cada fila represente una venta de un producto en un periodo específico.

La nueva estructura queda así:

- `CODIGO_PRODUCTO`
- `PRODUCTO`
- `PERIODO`
- `CANTIDAD_FACTURADA`


In [330]:
df_ventas_largo = df_ventas_original.melt(
    id_vars=["CODIGO_PRODUCTO", "PRODUCTO"],
    value_vars=columnas_meses,
    var_name="PERIODO",
    value_name="CANTIDAD_FACTURADA"
)

display(df_ventas_largo.head(20))


,CODIGO_PRODUCTO,PRODUCTO,PERIODO,CANTIDAD_FACTURADA
0,2542,[2542] ENVASE SADMAN TRANSP 105 ML PLATA CROMA...,ENERO 2026,NaN
1,2229,[2229] 180 FUNDAS SURTIDAS X 6 COLORES,ENERO 2026,144.0
2,2278,[2278] A MILKSHAKE PLEASE DM,ENERO 2026,246670.0
3,2834,[2834] ACEITE DE CASTOR P40 - FINDET ARH/52,ENERO 2026,NaN
4,0252,[0252] ACQUA FRESCA HM,ENERO 2026,87120.0
5,509,[509] ADDICTIVE VIBE HM,ENERO 2026,162420.0
6,0723,[0723] ADELINE DM,ENERO 2026,39370.0
7,400,[400] ADIVINA DM,ENERO 2026,25410.0
8,372,[372] ADREA HM,ENERO 2026,11060.0
9,291,[291] ADRIAN HM,ENERO 2026,120440.0


### Validación de tamaño después de transformar la tabla

Se revisa el número de filas y columnas después de pasar de formato ancho a formato largo.


In [331]:
print("Filas y columnas:", df_ventas_largo.shape)


Filas y columnas: (14232, 4)


## 10. Limpiar registros sin venta

Primero se eliminan las filas donde `CANTIDAD_FACTURADA` está vacía o nula.  
Estos registros no representan ventas reales y no aportan al informe.


In [332]:
df_ventas_largo = df_ventas_largo.dropna(subset=["CANTIDAD_FACTURADA"])

display(df_ventas_largo.head(20))


,CODIGO_PRODUCTO,PRODUCTO,PERIODO,CANTIDAD_FACTURADA
1,2229,[2229] 180 FUNDAS SURTIDAS X 6 COLORES,ENERO 2026,144.0
2,2278,[2278] A MILKSHAKE PLEASE DM,ENERO 2026,246670.0
4,0252,[0252] ACQUA FRESCA HM,ENERO 2026,87120.0
5,509,[509] ADDICTIVE VIBE HM,ENERO 2026,162420.0
6,0723,[0723] ADELINE DM,ENERO 2026,39370.0
7,400,[400] ADIVINA DM,ENERO 2026,25410.0
8,372,[372] ADREA HM,ENERO 2026,11060.0
9,291,[291] ADRIAN HM,ENERO 2026,120440.0
10,287,[287] ADRIANA DM,ENERO 2026,22240.0
11,2097,[2097] ADVENTUROUS SKIPPER HM,ENERO 2026,12080.0


### Validación de tamaño después de eliminar nulos

Se revisa nuevamente la cantidad de registros después de quitar ventas nulas.


In [333]:
print("Filas y columnas:", df_ventas_largo.shape)


Filas y columnas: (11247, 4)


## 11. Eliminar cantidades en cero o negativas

Para el análisis de ventas se conservan únicamente registros con cantidades facturadas mayores a cero.  
Los valores en cero o negativos pueden representar ajustes, devoluciones o registros que no corresponden al análisis principal de ventas.


In [334]:
df_ventas_largo = df_ventas_largo[
    df_ventas_largo["CANTIDAD_FACTURADA"] > 0
].copy()

display(df_ventas_largo.head(5))


,CODIGO_PRODUCTO,PRODUCTO,PERIODO,CANTIDAD_FACTURADA
1,2229,[2229] 180 FUNDAS SURTIDAS X 6 COLORES,ENERO 2026,144.0
2,2278,[2278] A MILKSHAKE PLEASE DM,ENERO 2026,246670.0
4,0252,[0252] ACQUA FRESCA HM,ENERO 2026,87120.0
5,509,[509] ADDICTIVE VIBE HM,ENERO 2026,162420.0
6,0723,[0723] ADELINE DM,ENERO 2026,39370.0


In [335]:
# ============================================================
# FUNCIÓN PARA NORMALIZAR CÓDIGOS DE PRODUCTO
# ============================================================

def normalizar_codigo_producto(df, columna_codigo="CODIGO_PRODUCTO", largo_codigo=4):
    """
    Convierte los códigos de producto a texto, elimina espacios,
    quita decimales innecesarios y agrega ceros a la izquierda.
    
    Ejemplo:
    1      -> 0001
    58     -> 0058
    495    -> 0495
    2608   -> 2608
    0001   -> 0001
    """
    
    df = df.copy()
    
    df[columna_codigo] = (
        df[columna_codigo]
        .astype("string")
        .str.strip()
        .str.replace(".0", "", regex=False)
        .str.zfill(largo_codigo)
    )
    
    return df

In [336]:
df_ventas_largo = normalizar_codigo_producto(
    df_ventas_largo,
    columna_codigo="CODIGO_PRODUCTO",
    largo_codigo=4
)

display(df_ventas_largo.head(5))

,CODIGO_PRODUCTO,PRODUCTO,PERIODO,CANTIDAD_FACTURADA
1,2229,[2229] 180 FUNDAS SURTIDAS X 6 COLORES,ENERO 2026,144.0
2,2278,[2278] A MILKSHAKE PLEASE DM,ENERO 2026,246670.0
4,0252,[0252] ACQUA FRESCA HM,ENERO 2026,87120.0
5,0509,[509] ADDICTIVE VIBE HM,ENERO 2026,162420.0
6,0723,[0723] ADELINE DM,ENERO 2026,39370.0


### Validación de estructura final de ventas limpias

Se revisa el tamaño y los tipos de datos después de limpiar cantidades nulas, negativas o en cero.


In [337]:
print("Filas y columnas:", df_ventas_largo.shape)
display(df_ventas_largo.dtypes)


Filas y columnas: (11179, 4)


CODIGO_PRODUCTO        string
PRODUCTO               string
PERIODO                   str
CANTIDAD_FACTURADA    float64
dtype: object

### Resumen de ventas por periodo

Esta validación permite confirmar que cada periodo conserve ventas después de la limpieza.  
También sirve como revisión rápida para detectar meses con valores muy bajos o inesperados.


In [338]:
print("Filas después de limpiar ventas:", df_ventas_largo.shape[0])

display(
    df_ventas_largo
    .groupby("PERIODO")["CANTIDAD_FACTURADA"]
    .sum()
    .reset_index()
)


Filas después de limpiar ventas: 11179


,PERIODO,CANTIDAD_FACTURADA
0,ABRIL 2026,34265475.0
1,ENERO 2026,23082792.0
2,FEBRERO 2026,22730809.0
3,JUNIO 2026,22477447.0
4,MARZO 2026,36378822.0
5,MAYO 2026,28687994.0


## 12. Preparar códigos para el cruce con la tabla maestra

Antes del cruce, se normaliza nuevamente `CODIGO_PRODUCTO` en ambas tablas.  
Esto reduce errores por espacios, formatos diferentes o códigos interpretados como números.


In [339]:
df_ventas_largo["CODIGO_PRODUCTO"] = (
    df_ventas_largo["CODIGO_PRODUCTO"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", "", regex=True)
)

df_maestro["CODIGO_PRODUCTO"] = (
    df_maestro["CODIGO_PRODUCTO"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", "", regex=True)
)


### Revisión rápida de la tabla maestra antes del cruce

Se muestran las columnas y las primeras filas de la tabla maestra para confirmar que está lista para cruzarse con las ventas.


In [340]:
print(df_maestro.columns)
display(df_maestro.head(20))

print(df_ventas_largo.columns)
display(df_ventas_largo.head(20))


Index(['CODIGO_PRODUCTO', 'NOMBRE_PRODUCTO', 'CLASIFICACION I',
       'CLASIFICACION II', 'CLASIFICACION III', 'CLASIFICACION IV'],
      dtype='str')


,CODIGO_PRODUCTO,NOMBRE_PRODUCTO,CLASIFICACION I,CLASIFICACION II,CLASIFICACION III,CLASIFICACION IV
0,2542,ENVASE SADMAN TRANSP 105 ML PLATA CROMADO X 54...,ENVASE,SADMAN,105 ML,54
1,2610,ENVASE SALVAJE 100 ML FUCSIA X 72 UND,ENVASE,SALVAJE,100 ML,72
2,2608,ENVASE SALVAJE 100 ML NEGRO X 72 UND,ENVASE,SALVAJE,100 ML,72
3,2229,180 FUNDAS SURTIDAS X 6 COLORES,FUNDA,PAQUETE,NO APLICA,NO APLICA
4,2278,A MILKSHAKE PLEASE DM,ESENCIA,ESENCIA NICHO,ARMAF,DM
5,2834,ACEITE DE CASTOR P40 - FINDET ARH/52,ACEITE,ACEITE,ACEITE,ACEITE
6,0252,ACQUA FRESCA HM,ESENCIA,ESENCIA GENERAL,ARMANI,HM
7,0509,ADDICTIVE VIBE HM,ESENCIA,ESENCIA GENERAL,PACO RABANNE,HM
8,0723,ADELINE DM,ESENCIA,ESENCIA NICHO,PARFUMS DE MARLY,DM
9,0400,ADIVINA DM,ESENCIA,ESENCIA GENERAL,GUESS,DM


Index(['CODIGO_PRODUCTO', 'PRODUCTO', 'PERIODO', 'CANTIDAD_FACTURADA'], dtype='str')


,CODIGO_PRODUCTO,PRODUCTO,PERIODO,CANTIDAD_FACTURADA
1,2229,[2229] 180 FUNDAS SURTIDAS X 6 COLORES,ENERO 2026,144.0
2,2278,[2278] A MILKSHAKE PLEASE DM,ENERO 2026,246670.0
4,0252,[0252] ACQUA FRESCA HM,ENERO 2026,87120.0
5,0509,[509] ADDICTIVE VIBE HM,ENERO 2026,162420.0
6,0723,[0723] ADELINE DM,ENERO 2026,39370.0
7,0400,[400] ADIVINA DM,ENERO 2026,25410.0
8,0372,[372] ADREA HM,ENERO 2026,11060.0
9,0291,[291] ADRIAN HM,ENERO 2026,120440.0
10,0287,[287] ADRIANA DM,ENERO 2026,22240.0
11,2097,[2097] ADVENTUROUS SKIPPER HM,ENERO 2026,12080.0


## 13. Cruzar ventas con tabla maestra

Se realiza un cruce por `CODIGO_PRODUCTO` para agregar a cada venta el nombre y las clasificaciones del producto.

Se usa `how="left"` para conservar inicialmente todas las ventas, incluso si algún producto no se encuentra en la tabla maestra.  
Después se validan y separan los productos no clasificados.


In [341]:
df_ventas_powerbi = df_ventas_largo.merge(
    df_maestro,
    on="CODIGO_PRODUCTO",
    how="left"
)

display(df_ventas_powerbi.head(10))


,CODIGO_PRODUCTO,PRODUCTO,PERIODO,CANTIDAD_FACTURADA,NOMBRE_PRODUCTO,CLASIFICACION I,CLASIFICACION II,CLASIFICACION III,CLASIFICACION IV
0,2229,[2229] 180 FUNDAS SURTIDAS X 6 COLORES,ENERO 2026,144.0,180 FUNDAS SURTIDAS X 6 COLORES,FUNDA,PAQUETE,NO APLICA,NO APLICA
1,2278,[2278] A MILKSHAKE PLEASE DM,ENERO 2026,246670.0,A MILKSHAKE PLEASE DM,ESENCIA,ESENCIA NICHO,ARMAF,DM
2,0252,[0252] ACQUA FRESCA HM,ENERO 2026,87120.0,ACQUA FRESCA HM,ESENCIA,ESENCIA GENERAL,ARMANI,HM
3,0509,[509] ADDICTIVE VIBE HM,ENERO 2026,162420.0,ADDICTIVE VIBE HM,ESENCIA,ESENCIA GENERAL,PACO RABANNE,HM
4,0723,[0723] ADELINE DM,ENERO 2026,39370.0,ADELINE DM,ESENCIA,ESENCIA NICHO,PARFUMS DE MARLY,DM
5,0400,[400] ADIVINA DM,ENERO 2026,25410.0,ADIVINA DM,ESENCIA,ESENCIA GENERAL,GUESS,DM
6,0372,[372] ADREA HM,ENERO 2026,11060.0,ADREA HM,ESENCIA,ESENCIA GENERAL,ENRIQUE IGLESIAS,HM
7,0291,[291] ADRIAN HM,ENERO 2026,120440.0,ADRIAN HM,ESENCIA,ESENCIA GENERAL,CAROLINA HERRERA,HM
8,0287,[287] ADRIANA DM,ENERO 2026,22240.0,ADRIANA DM,ESENCIA,ESENCIA GENERAL,CAROLINA HERRERA,DM
9,2097,[2097] ADVENTUROUS SKIPPER HM,ENERO 2026,12080.0,ADVENTUROUS SKIPPER HM,ESENCIA,ESENCIA GENERAL,NAUTICA,HM


### Validación de columnas después del cruce

Se revisan las columnas resultantes para confirmar que el cruce agregó correctamente los campos de la tabla maestra.


In [342]:
print(df_ventas_powerbi.columns)


Index(['CODIGO_PRODUCTO', 'PRODUCTO', 'PERIODO', 'CANTIDAD_FACTURADA',
       'NOMBRE_PRODUCTO', 'CLASIFICACION I', 'CLASIFICACION II',
       'CLASIFICACION III', 'CLASIFICACION IV'],
      dtype='str')


## 14. Identificar productos no clasificados

Los productos no clasificados son aquellos que aparecen en ventas, pero no tienen coincidencia en la tabla maestra.  
Estos productos se separan para poder actualizar la tabla maestra posteriormente.


In [343]:
productos_no_clasificados = (
    df_ventas_powerbi[
        df_ventas_powerbi["NOMBRE_PRODUCTO"].isna()
    ][["CODIGO_PRODUCTO", "PRODUCTO"]]
    .drop_duplicates()
)

print("Cantidad de productos no clasificados:", productos_no_clasificados.shape[0])
display(productos_no_clasificados)


Cantidad de productos no clasificados: 4


,CODIGO_PRODUCTO,PRODUCTO
1806,1553,[1553] Transporte
5543,2823,[2823] TINA PLASTICA 200 L
9788,2611,[2611] ENVASE SALVAJE 100 ML AZUL REY X 72 UND
9794,2612,[2612] ENVASE SALVAJE 100 ML TRANSP DORADO X 7...


### Guardar productos no clasificados

Se genera un archivo auxiliar con los productos que no pudieron clasificarse.  
Este archivo sirve para revisar y completar la tabla maestra en futuras actualizaciones.


In [344]:
productos_no_clasificados.to_excel(
    ruta_productos_no_clasificados,
    sheet_name="Productos",
    index=False
)

print("Archivo de productos no clasificados generado:", ruta_productos_no_clasificados)


Archivo de productos no clasificados generado: ..\Data\Tabla_Productos_No_Clasificados.xlsx


### Validación de registros no clasificados antes de eliminarlos

Además del listado único de productos, se guardan todos los registros de ventas que quedaron sin clasificación.  
Esto permite revisar el impacto real de esos productos en la tabla de ventas.


In [345]:
df_registros_no_clasificados = df_ventas_powerbi[
    df_ventas_powerbi["NOMBRE_PRODUCTO"].isna()
].copy()

print("Registros de ventas no clasificados:", df_registros_no_clasificados.shape[0])
display(df_registros_no_clasificados.head(5))


Registros de ventas no clasificados: 8


,CODIGO_PRODUCTO,PRODUCTO,PERIODO,CANTIDAD_FACTURADA,NOMBRE_PRODUCTO,CLASIFICACION I,CLASIFICACION II,CLASIFICACION III,CLASIFICACION IV
1806,1553,[1553] Transporte,ENERO 2026,7.0,NaN,NaN,NaN,NaN,NaN
3676,1553,[1553] Transporte,FEBRERO 2026,1.0,NaN,NaN,NaN,NaN,NaN
5543,2823,[2823] TINA PLASTICA 200 L,MARZO 2026,1.0,NaN,NaN,NaN,NaN,NaN
5565,1553,[1553] Transporte,MARZO 2026,6.0,NaN,NaN,NaN,NaN,NaN
7449,1553,[1553] Transporte,ABRIL 2026,5.0,NaN,NaN,NaN,NaN,NaN


## 15. Eliminar registros no clasificados

Después de separarlos para revisión, se eliminan del archivo final los registros que no tienen `NOMBRE_PRODUCTO`.  
Esto evita llevar a Power BI productos sin clasificación.


In [346]:
df_ventas_powerbi = df_ventas_powerbi[
    df_ventas_powerbi["NOMBRE_PRODUCTO"].notna()
].copy()

# Reiniciar índice después de eliminar registros no clasificados
df_ventas_powerbi = df_ventas_powerbi.reset_index(drop=True)

display(df_ventas_powerbi.head())


,CODIGO_PRODUCTO,PRODUCTO,PERIODO,CANTIDAD_FACTURADA,NOMBRE_PRODUCTO,CLASIFICACION I,CLASIFICACION II,CLASIFICACION III,CLASIFICACION IV
0,2229,[2229] 180 FUNDAS SURTIDAS X 6 COLORES,ENERO 2026,144.0,180 FUNDAS SURTIDAS X 6 COLORES,FUNDA,PAQUETE,NO APLICA,NO APLICA
1,2278,[2278] A MILKSHAKE PLEASE DM,ENERO 2026,246670.0,A MILKSHAKE PLEASE DM,ESENCIA,ESENCIA NICHO,ARMAF,DM
2,0252,[0252] ACQUA FRESCA HM,ENERO 2026,87120.0,ACQUA FRESCA HM,ESENCIA,ESENCIA GENERAL,ARMANI,HM
3,0509,[509] ADDICTIVE VIBE HM,ENERO 2026,162420.0,ADDICTIVE VIBE HM,ESENCIA,ESENCIA GENERAL,PACO RABANNE,HM
4,0723,[0723] ADELINE DM,ENERO 2026,39370.0,ADELINE DM,ESENCIA,ESENCIA NICHO,PARFUMS DE MARLY,DM


### Validación posterior a la eliminación de no clasificados

Se confirma que no queden registros sin clasificación en la tabla final.


In [347]:
registros_sin_clasificar = df_ventas_powerbi["NOMBRE_PRODUCTO"].isna().sum()

print("Registros sin clasificar:", registros_sin_clasificar)
print("Filas finales:", df_ventas_powerbi.shape[0])


Registros sin clasificar: 0
Filas finales: 11178


### Validación de campos de clasificación

Se revisan los campos principales de producto y clasificación para identificar valores faltantes antes de ordenar la tabla final.


In [348]:
display(
    df_ventas_powerbi[
        [
            "NOMBRE_PRODUCTO",
            "CLASIFICACION I",
            "CLASIFICACION II",
            "CLASIFICACION III",
            "CLASIFICACION IV"
        ]
    ].isna().sum()
)


NOMBRE_PRODUCTO      0
CLASIFICACION I      0
CLASIFICACION II     0
CLASIFICACION III    0
CLASIFICACION IV     0
dtype: int64

## 16. Ordenar columnas finales para Power BI

Se deja la tabla final con las columnas necesarias y en un orden más práctico para construir visualizaciones en Power BI.


In [349]:
df_ventas_powerbi = df_ventas_powerbi[
    [
        "CODIGO_PRODUCTO",
        "NOMBRE_PRODUCTO",
        "CLASIFICACION I",
        "CLASIFICACION II",
        "CLASIFICACION III",
        "CLASIFICACION IV",
        "PERIODO",
        "CANTIDAD_FACTURADA"
    ]
]

display(df_ventas_powerbi.head())


,CODIGO_PRODUCTO,NOMBRE_PRODUCTO,CLASIFICACION I,CLASIFICACION II,CLASIFICACION III,CLASIFICACION IV,PERIODO,CANTIDAD_FACTURADA
0,2229,180 FUNDAS SURTIDAS X 6 COLORES,FUNDA,PAQUETE,NO APLICA,NO APLICA,ENERO 2026,144.0
1,2278,A MILKSHAKE PLEASE DM,ESENCIA,ESENCIA NICHO,ARMAF,DM,ENERO 2026,246670.0
2,0252,ACQUA FRESCA HM,ESENCIA,ESENCIA GENERAL,ARMANI,HM,ENERO 2026,87120.0
3,0509,ADDICTIVE VIBE HM,ESENCIA,ESENCIA GENERAL,PACO RABANNE,HM,ENERO 2026,162420.0
4,0723,ADELINE DM,ESENCIA,ESENCIA NICHO,PARFUMS DE MARLY,DM,ENERO 2026,39370.0


### Validación de nulos en la tabla final

Esta revisión confirma si la tabla final todavía contiene valores nulos después de todo el proceso de limpieza y clasificación.


In [350]:
display(df_ventas_powerbi.isnull().sum())


CODIGO_PRODUCTO       0
NOMBRE_PRODUCTO       0
CLASIFICACION I       0
CLASIFICACION II      0
CLASIFICACION III     0
CLASIFICACION IV      0
PERIODO               0
CANTIDAD_FACTURADA    0
dtype: int64

### Validación de tipos de datos finales

Se revisan los tipos de datos de la tabla final para confirmar que `CANTIDAD_FACTURADA` pueda ser usada como medida numérica en Power BI.


In [351]:
display(df_ventas_powerbi.dtypes)


CODIGO_PRODUCTO        string
NOMBRE_PRODUCTO           str
CLASIFICACION I           str
CLASIFICACION II          str
CLASIFICACION III         str
CLASIFICACION IV          str
PERIODO                   str
CANTIDAD_FACTURADA    float64
dtype: object

## 17. Asegurar formato numérico de cantidad facturada

Se convierte `CANTIDAD_FACTURADA` a formato numérico como último control antes de exportar el archivo.


In [352]:
df_ventas_powerbi["CANTIDAD_FACTURADA"] = pd.to_numeric(
    df_ventas_powerbi["CANTIDAD_FACTURADA"],
    errors="coerce"
)


## 18. Ordenar registros finales

La tabla se ordena por periodo, clasificación principal y nombre del producto.  
Esto facilita la revisión manual del archivo exportado.


In [353]:
df_ventas_powerbi = df_ventas_powerbi.sort_values(
    by=["PERIODO", "CLASIFICACION I", "NOMBRE_PRODUCTO"]
).reset_index(drop=True)

display(df_ventas_powerbi.head())


,CODIGO_PRODUCTO,NOMBRE_PRODUCTO,CLASIFICACION I,CLASIFICACION II,CLASIFICACION III,CLASIFICACION IV,PERIODO,CANTIDAD_FACTURADA
0,0001,ALCOHOL EXTRA NEUTRO.,ALCOHOL,ALCOHOL,ALCOHOL,ALCOHOL,ABRIL 2026,29880.0
1,2780,CAJA CILINDRICA BLANCA 100 ML X 120 UND,CAJA,CILINDRICA,100 ML,NO APLICA,ABRIL 2026,120.0
2,0008,CAJA PARA CILINDRO 1OZ DORADO,CAJA,CILINDRO,1 OZ,NO APLICA,ABRIL 2026,1100.0
3,0009,CAJA PARA CILINDRO 1OZ FUCSIA,CAJA,CILINDRO,1 OZ,NO APLICA,ABRIL 2026,1700.0
4,0006,CAJA PARA CILINDRO 1OZ NEGRA,CAJA,CILINDRO,1 OZ,NO APLICA,ABRIL 2026,1700.0


## 19. Crear un archivo Excel por cada periodo

En este paso se toma la columna `PERIODO` y se genera un archivo independiente por cada mes encontrado en el informe.

Ejemplo:

- `Ventas_Limpias_MARZO_2026.xlsx`
- `Ventas_Limpias_FEBRERO_2025.xlsx`

Todos los archivos mensuales quedan guardados en la carpeta `Informes limpios`.

Si el archivo mensual ya existe, el notebook se detiene para evitar sobrescribir información sin validar.

In [354]:
# ============================================================
# CREAR ARCHIVOS LIMPIOS POR PERIODO
# ============================================================

if COLUMNA_PERIODO not in df_ventas_powerbi.columns:
    raise ValueError(
        f"No existe la columna '{COLUMNA_PERIODO}' en df_ventas_powerbi. "
        f"Columnas disponibles: {list(df_ventas_powerbi.columns)}"
    )

# Normalizar el periodo para evitar problemas por espacios
df_ventas_powerbi[COLUMNA_PERIODO] = (
    df_ventas_powerbi[COLUMNA_PERIODO]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Validar periodos vacíos
periodos_vacios = df_ventas_powerbi[
    df_ventas_powerbi[COLUMNA_PERIODO].isna()
    | (df_ventas_powerbi[COLUMNA_PERIODO] == "")
    | (df_ventas_powerbi[COLUMNA_PERIODO] == "NAN")
]

if not periodos_vacios.empty:
    print("Registros con periodo vacío:")
    display(periodos_vacios.head(20))
    raise ValueError("Hay registros sin periodo. Revisa el informe antes de continuar.")

# Función para convertir el periodo en un nombre de archivo seguro
def limpiar_nombre_archivo(texto):
    texto = str(texto).strip().upper()
    texto = unicodedata.normalize("NFKD", texto).encode("ASCII", "ignore").decode("utf-8")
    texto = re.sub(r"[^\w]+", "_", texto)
    texto = re.sub(r"_+", "_", texto).strip("_")
    return texto

periodos_detectados = sorted(df_ventas_powerbi[COLUMNA_PERIODO].dropna().unique())

print("Periodos detectados en el informe:")
display(pd.DataFrame({"PERIODO": periodos_detectados}))

# Validar si ya existen archivos de esos periodos
archivos_existentes = []

for periodo in periodos_detectados:
    periodo_archivo = limpiar_nombre_archivo(periodo)
    ruta_periodo = CARPETA_SALIDA_MENSUAL / f"Ventas_Limpias_{periodo_archivo}.xlsx"

    if ruta_periodo.exists():
        archivos_existentes.append(ruta_periodo)

if archivos_existentes and not PERMITIR_SOBREESCRITURA:
    print("ALERTA: ya existen archivos para uno o más periodos.")
    print("El proceso se detuvo para evitar sobrescribir información existente.")
    print()
    print("Archivos encontrados:")

    for archivo in archivos_existentes:
        print(f"- {archivo}")

    raise FileExistsError(
        "Valida si deseas reemplazar esos archivos. "
        "Para continuar, cambia PERMITIR_SOBREESCRITURA = True."
    )

# Exportar un Excel por cada periodo
resumen_archivos_mensuales = []

for periodo, df_periodo in df_ventas_powerbi.groupby(COLUMNA_PERIODO):
    periodo_archivo = limpiar_nombre_archivo(periodo)
    ruta_periodo = CARPETA_SALIDA_MENSUAL / f"Ventas_Limpias_{periodo_archivo}.xlsx"

    df_periodo = df_periodo.copy()
    df_periodo = df_periodo.sort_values(
        by=["CLASIFICACION I", "NOMBRE_PRODUCTO"]
    ).reset_index(drop=True)

    with pd.ExcelWriter(ruta_periodo, engine="openpyxl") as writer:
        df_periodo.to_excel(
            writer,
            sheet_name="Ventas_Limpias",
            index=False
        )

    resumen_archivos_mensuales.append({
        "PERIODO": periodo,
        "REGISTROS": len(df_periodo),
        "ARCHIVO_GENERADO": ruta_periodo.name
    })

df_resumen_archivos_mensuales = pd.DataFrame(resumen_archivos_mensuales)

print("Archivos mensuales generados correctamente en:", CARPETA_SALIDA_MENSUAL)
display(df_resumen_archivos_mensuales)


Periodos detectados en el informe:


,PERIODO
0,ABRIL 2026
1,ENERO 2026
2,FEBRERO 2026
3,JUNIO 2026
4,MARZO 2026
5,MAYO 2026


Archivos mensuales generados correctamente en: ..\Informes Limpios


,PERIODO,REGISTROS,ARCHIVO_GENERADO
0,ABRIL 2026,1883,Ventas_Limpias_ABRIL_2026.xlsx
1,ENERO 2026,1895,Ventas_Limpias_ENERO_2026.xlsx
2,FEBRERO 2026,1870,Ventas_Limpias_FEBRERO_2026.xlsx
3,JUNIO 2026,1790,Ventas_Limpias_JUNIO_2026.xlsx
4,MARZO 2026,1885,Ventas_Limpias_MARZO_2026.xlsx
5,MAYO 2026,1855,Ventas_Limpias_MAYO_2026.xlsx


## 20. Consolidar todos los archivos mensuales para Power BI

Después de crear los Excel por periodo, este paso lee todos los archivos de la carpeta `Informes limpios` y crea un único archivo consolidado.

El archivo final queda guardado en la carpeta `Resultado` con el nombre:

`Ventas_Limpias_PowerBI.xlsx`

In [355]:
# ============================================================
# CONSOLIDAR TODOS LOS ARCHIVOS MENSUALES PARA POWER BI
# ============================================================

archivos_mensuales = sorted(CARPETA_SALIDA_MENSUAL.glob("Ventas_Limpias_*.xlsx"))

if not archivos_mensuales:
    raise ValueError(
        f"No se encontraron archivos mensuales en la carpeta: {CARPETA_SALIDA_MENSUAL}"
    )

dataframes = []

for archivo in archivos_mensuales:
    
    # Leer cada archivo mensual asegurando que CODIGO_PRODUCTO sea texto
    df_temp = pd.read_excel(
        archivo,
        sheet_name="Ventas_Limpias",
        dtype={
            "CODIGO_PRODUCTO": str
        }
    )
    
    # Normalizar código por seguridad
    df_temp["CODIGO_PRODUCTO"] = (
        df_temp["CODIGO_PRODUCTO"]
        .astype("string")
        .str.strip()
        .str.replace(".0", "", regex=False)
        .str.zfill(4)
    )
    
    dataframes.append(df_temp)

# Unir todos los archivos mensuales
df_consolidado_powerbi = pd.concat(dataframes, ignore_index=True)

# Eliminar columna origen si existe
if "ARCHIVO_ORIGEN" in df_consolidado_powerbi.columns:
    df_consolidado_powerbi = df_consolidado_powerbi.drop(columns=["ARCHIVO_ORIGEN"])

# Normalizar nuevamente después de unir todo
df_consolidado_powerbi["CODIGO_PRODUCTO"] = (
    df_consolidado_powerbi["CODIGO_PRODUCTO"]
    .astype("string")
    .str.strip()
    .str.replace(".0", "", regex=False)
    .str.zfill(4)
)

# Ordenar solo por periodo
df_consolidado_powerbi = df_consolidado_powerbi.sort_values(
    by=["PERIODO"]
).reset_index(drop=True)

# Guardar consolidado final para Power BI
with pd.ExcelWriter(ruta_salida_powerbi, engine="openpyxl") as writer:
    df_consolidado_powerbi.to_excel(
        writer,
        sheet_name="Ventas_Limpias",
        index=False
    )

print("Archivo consolidado generado correctamente:")
print(ruta_salida_powerbi)
print("Total registros consolidados:", len(df_consolidado_powerbi))

display(
    df_consolidado_powerbi
    .groupby("PERIODO")["CANTIDAD_FACTURADA"]
    .agg(["count", "sum"])
    .reset_index()
)


Archivo consolidado generado correctamente:
..\Resultado\Ventas_Limpias_PowerBI.xlsx
Total registros consolidados: 50233


,PERIODO,count,sum
0,ABRIL 2024,1359,22106719.0
1,ABRIL 2025,1802,21608393.0
2,ABRIL 2026,1883,34332760.0
3,AGOSTO 2024,1419,20042234.0
4,AGOSTO 2025,1874,23409300.0
5,DICIEMBRE 2024,1668,26871044.0
6,DICIEMBRE 2025,1948,29544453.0
7,ENERO 2024,1027,4030633.0
8,ENERO 2025,1682,22134324.0
9,ENERO 2026,1895,23156875.0


## 20. Resultado del proceso

Al finalizar el notebook se obtienen dos archivos:

1. **Ventas limpias para Power BI:** contiene únicamente ventas válidas y productos clasificados.
2. **Productos no clasificados:** contiene los productos que deben revisarse o agregarse a la tabla maestra.

Con este flujo se conserva la trazabilidad del proceso y se mantienen las validaciones necesarias para detectar problemas en cada ejecución mensual.
